In [0]:
# Databricks notebook source
from datetime import datetime, timezone
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

RUN_ID = f"uc16_ai_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
CHECKED_AT = datetime.now(timezone.utc)

messages = spark.table(
    "poc_health_no_show.gold.outreach_message_variants"
)

unsafe_terms = [
    "diagnos",
    "treatment",
    "medic",
    "dose",
    "disease",
    "condition",
    "prescri"
]

results = []

for row in messages.collect():
    appointment_id = row["appointment_id"]
    variant = row["variant"]
    message = row["message_text"] or ""
    message_length = len(message)

    checks = [
        (
            "message_not_empty",
            message.strip() != ""
        ),
        (
            "message_length_valid",
            message_length <= 500
        ),
        (
            "no_unsafe_clinical_terms",
            not any(term in message.lower() for term in unsafe_terms)
        ),
        (
            "no_patient_id_in_message",
            appointment_id not in message
        )
    ]

    for check_name, passed in checks:
        results.append((
            RUN_ID,
            appointment_id,
            variant,
            check_name,
            "PASS" if passed else "FAIL",
            message_length,
            CHECKED_AT
        ))

schema = """
  run_id STRING,
  appointment_id STRING,
  variant STRING,
  check_name STRING,
  status STRING,
  message_length INT,
  checked_at TIMESTAMP
"""

quality_df = spark.createDataFrame(results, schema)

quality_df.write.mode("append").saveAsTable(
    "poc_health_no_show.ops.ai_message_quality_results"
)

failed = [
    row for row in results if row[4] == "FAIL"
]

if failed:
    raise Exception(f"AI message quality checks failed: {failed}")

print("All AI message-quality checks passed.")